1. MTD Performance vs Previous MTD: 
Month-to-date revenue and number of completed orders compared with previous month-to-date, by store and manager. 

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.kpi_mtd_performance AS
WITH latest_month_data AS (
  SELECT MAX(invoice_date) as max_date
  FROM automobile_repair.gold.fact_revenue
),
current_month AS (
  SELECT
    f.store_id,
    s.store_name,
    s.manager_id,
    s.manager_name,
    YEAR(f.invoice_date) as year,
    MONTH(f.invoice_date) as month,
    SUM(f.invoice_amount) as mtd_revenue,
    COUNT(DISTINCT f.order_id) as mtd_orders
  FROM automobile_repair.gold.fact_revenue f
  JOIN automobile_repair.gold.dim_store s ON f.store_id = s.store_id
  CROSS JOIN latest_month_data lm
  WHERE YEAR(f.invoice_date) = YEAR(lm.max_date)
    AND MONTH(f.invoice_date) = MONTH(lm.max_date)
  GROUP BY f.store_id, s.store_name, s.manager_id, s.manager_name, YEAR(f.invoice_date), MONTH(f.invoice_date)
),
previous_month AS (
  SELECT
    f.store_id,
    SUM(f.invoice_amount) as prev_mtd_revenue,
    COUNT(DISTINCT f.order_id) as prev_mtd_orders
  FROM automobile_repair.gold.fact_revenue f
  CROSS JOIN latest_month_data lm
  WHERE YEAR(f.invoice_date) = YEAR(ADD_MONTHS(lm.max_date, -1))
    AND MONTH(f.invoice_date) = MONTH(ADD_MONTHS(lm.max_date, -1))
  GROUP BY f.store_id
)
SELECT
  c.store_id,
  c.store_name,
  c.manager_id,
  c.manager_name,
  c.year,
  c.month,
  c.mtd_revenue,
  c.mtd_orders,
  COALESCE(p.prev_mtd_revenue, 0) as prev_month_revenue,
  COALESCE(p.prev_mtd_orders, 0) as prev_month_orders,
  c.mtd_revenue - COALESCE(p.prev_mtd_revenue, 0) as revenue_change,
  ROUND(((c.mtd_revenue - COALESCE(p.prev_mtd_revenue, 0)) / NULLIF(p.prev_mtd_revenue, 0)) * 100, 2) as revenue_change_pct,
  c.mtd_orders - COALESCE(p.prev_mtd_orders, 0) as orders_change,
  ROUND(((c.mtd_orders - COALESCE(p.prev_mtd_orders, 0)) / NULLIF(p.prev_mtd_orders, 0)) * 100, 2) as orders_change_pct,
  current_timestamp() as created_at
FROM current_month c
LEFT JOIN previous_month p ON c.store_id = p.store_id
ORDER BY c.store_id

2. Average Days in Shop: 
Average days a vehicle spends in the shop, by store and service type. 

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.kpi_avg_days_in_shop AS
SELECT
  f.store_id,
  s.store_name,
  f.service_type,
  COUNT(f.order_id) as total_orders,
  ROUND(AVG(f.days_in_shop), 2) as avg_days_in_shop,
  MIN(f.days_in_shop) as min_days_in_shop,
  MAX(f.days_in_shop) as max_days_in_shop,
  current_timestamp() as created_at
FROM automobile_repair.gold.fact_order_details f
JOIN automobile_repair.gold.dim_store s ON f.store_id = s.store_id
WHERE f.days_in_shop IS NOT NULL
GROUP BY f.store_id, s.store_name, f.service_type
ORDER BY f.store_id, avg_days_in_shop DESC

3. Survey Coverage: 
For each store, number of surveys sent versus number of surveys responded. 

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.kpi_survey_coverage AS
SELECT
  o.store_id,
  s.store_name,
  COUNT(DISTINCT o.order_id) as total_orders,
  COUNT(DISTINCT cs.survey_id) as surveys_sent,
  SUM(CASE WHEN cs.responded_flag = 1 THEN 1 ELSE 0 END) as surveys_responded,
  ROUND((COUNT(DISTINCT cs.survey_id) / COUNT(DISTINCT o.order_id)) * 100, 2) as survey_sent_rate,
  ROUND((SUM(CASE WHEN cs.responded_flag = 1 THEN 1 ELSE 0 END) / NULLIF(COUNT(DISTINCT cs.survey_id), 0)) * 100, 2) as response_rate,
  current_timestamp() as created_at
FROM automobile_repair.gold.fact_order_details o
JOIN automobile_repair.gold.dim_store s ON o.store_id = s.store_id
LEFT JOIN automobile_repair.gold.fact_customer_survey cs ON o.order_id = cs.order_id
GROUP BY o.store_id, s.store_name
ORDER BY response_rate DESC

4. Survey Scores Summary: 
Overall customer survey metrics by store and ranking of stores by overall satisfaction. 

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.kpi_survey_scores AS
SELECT
  o.store_id,
  s.store_name,
  COUNT(cs.survey_id) as total_responses,
  ROUND(AVG(cs.delivered_on_time_rating), 2) as avg_delivered_on_time,
  ROUND(AVG(cs.work_quality_rating), 2) as avg_work_quality,
  ROUND(AVG(cs.cleanliness_rating), 2) as avg_cleanliness,
  ROUND(AVG(cs.communication_rating), 2) as avg_communication,
  ROUND(AVG(cs.overall_satisfaction_rating), 2) as avg_overall_satisfaction,
  ROUND(AVG(cs.average_rating), 2) as avg_all_ratings,
  RANK() OVER (ORDER BY AVG(cs.average_rating) DESC) as satisfaction_rank,
  current_timestamp() as created_at
FROM automobile_repair.gold.fact_customer_survey cs
JOIN automobile_repair.gold.fact_order_details o ON cs.order_id = o.order_id
JOIN automobile_repair.gold.dim_store s ON o.store_id = s.store_id
WHERE cs.responded_flag = 1
GROUP BY o.store_id, s.store_name
ORDER BY avg_all_ratings DESC

5.Revenue vs Budget: 
Monthly revenue compared against budget by manager, and ranking of managers by budget achievement. 

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.kpi_revenue_vs_budget AS
SELECT
  r.manager_id,
  r.manager_name,
  r.invoice_year as year,
  r.invoice_month as month,
  SUM(r.invoice_amount) as actual_revenue,
  MAX(b.budget_amount) as budget_amount,
  SUM(r.invoice_amount) - MAX(b.budget_amount) as variance,
  ROUND(((SUM(r.invoice_amount) - MAX(b.budget_amount)) / NULLIF(MAX(b.budget_amount), 0)) * 100, 2) as variance_pct,
  CASE
    WHEN SUM(r.invoice_amount) >= MAX(b.budget_amount) THEN 'Met or Exceeded'
    ELSE 'Below Budget'
  END as status,
  RANK() OVER (PARTITION BY r.invoice_year, r.invoice_month ORDER BY SUM(r.invoice_amount) DESC) as revenue_rank,
  current_timestamp() as created_at
FROM automobile_repair.gold.fact_revenue r
LEFT JOIN automobile_repair.silver.budget b 
  ON r.store_id = b.store_id 
  AND r.invoice_year = b.budget_year 
  AND r.invoice_month = b.budget_month
GROUP BY r.manager_id, r.manager_name, r.invoice_year, r.invoice_month
ORDER BY year DESC, month DESC, actual_revenue DESC

6. Top Technicians by Completion Time Accuracy: 
For each store, top 5 technicians based on accuracy of meeting promised completion dates.

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.kpi_top_technicians AS
WITH technician_performance AS (
  SELECT
    f.store_id,
    s.store_name,
    f.technician_id,
    t.technician_name,
    COUNT(f.order_id) as total_orders,
    AVG(f.delivery_accuracy_days) as avg_delivery_accuracy,
    SUM(CASE WHEN f.delivery_accuracy_days <= 0 THEN 1 ELSE 0 END) as on_time_deliveries,
    ROUND((SUM(CASE WHEN f.delivery_accuracy_days <= 0 THEN 1 ELSE 0 END) / COUNT(f.order_id)) * 100, 2) as on_time_pct,
    ROW_NUMBER() OVER (PARTITION BY f.store_id ORDER BY (SUM(CASE WHEN f.delivery_accuracy_days <= 0 THEN 1 ELSE 0 END) / COUNT(f.order_id)) DESC) as store_rank
  FROM automobile_repair.gold.fact_order_details f
  JOIN automobile_repair.gold.dim_store s ON f.store_id = s.store_id
  JOIN automobile_repair.gold.dim_technician t ON f.technician_id = t.technician_id
  WHERE f.delivery_accuracy_days IS NOT NULL
  GROUP BY f.store_id, s.store_name, f.technician_id, t.technician_name
)
SELECT
  store_id,
  store_name,
  technician_id,
  technician_name,
  total_orders,
  avg_delivery_accuracy,
  on_time_deliveries,
  on_time_pct,
  store_rank,
  current_timestamp() as created_at
FROM technician_performance
WHERE store_rank <= 5
ORDER BY store_id, store_rank

7.Year-to-Date Revenue Growth: 
Year-to-date revenue compared with previous year YTD, and ranking of stores by YTD growth. 

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.kpi_ytd_revenue_growth AS
WITH current_ytd AS (
  SELECT
    f.store_id,
    s.store_name,
    YEAR(f.invoice_date) as year,
    SUM(f.invoice_amount) as ytd_revenue,
    COUNT(DISTINCT f.order_id) as ytd_orders
  FROM automobile_repair.gold.fact_revenue f
  JOIN automobile_repair.gold.dim_store s ON f.store_id = s.store_id
  WHERE YEAR(f.invoice_date) = YEAR(CURRENT_DATE())
  GROUP BY f.store_id, s.store_name, YEAR(f.invoice_date)
),
prior_ytd AS (
  SELECT
    f.store_id,
    SUM(f.invoice_amount) as prior_ytd_revenue,
    COUNT(DISTINCT f.order_id) as prior_ytd_orders
  FROM automobile_repair.gold.fact_revenue f
  WHERE YEAR(f.invoice_date) = YEAR(CURRENT_DATE()) - 1
  GROUP BY f.store_id
)
SELECT
  c.store_id,
  c.store_name,
  c.year,
  c.ytd_revenue,
  c.ytd_orders,
  COALESCE(p.prior_ytd_revenue, 0) as prior_year_ytd_revenue,
  COALESCE(p.prior_ytd_orders, 0) as prior_year_ytd_orders,
  c.ytd_revenue - COALESCE(p.prior_ytd_revenue, 0) as revenue_growth,
  ROUND(((c.ytd_revenue - COALESCE(p.prior_ytd_revenue, 0)) / NULLIF(p.prior_ytd_revenue, 0)) * 100, 2) as revenue_growth_pct,
  current_timestamp() as created_at
FROM current_ytd c
LEFT JOIN prior_ytd p ON c.store_id = p.store_id
ORDER BY revenue_growth_pct DESC

8. Stage-wise Day Cycle Time: 
Average days spent in each stage (vehicle-in to work-start, work-start to completion, completion to delivery), by store and service type. 

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.kpi_cycle_time AS
SELECT
  f.store_id,
  s.store_name,
  f.service_type,
  COUNT(f.order_id) as total_orders,
  ROUND(AVG(DATEDIFF(f.actual_work_start_datetime, f.vehicle_in_datetime)), 2) as avg_wait_to_start_days,
  ROUND(AVG(DATEDIFF(f.actual_completion_datetime, f.actual_work_start_datetime)), 2) as avg_work_duration_days,
  ROUND(AVG(DATEDIFF(f.vehicle_out_datetime, f.actual_completion_datetime)), 2) as avg_completion_to_delivery_days,
  ROUND(AVG(DATEDIFF(f.vehicle_out_datetime, f.vehicle_in_datetime)), 2) as avg_total_cycle_days,
  current_timestamp() as created_at
FROM automobile_repair.gold.fact_order_details f
JOIN automobile_repair.gold.dim_store s ON f.store_id = s.store_id
WHERE f.vehicle_in_datetime IS NOT NULL
  AND f.actual_work_start_datetime IS NOT NULL
  AND f.actual_completion_datetime IS NOT NULL
  AND f.vehicle_out_datetime IS NOT NULL
GROUP BY f.store_id, s.store_name, f.service_type
ORDER BY f.store_id, f.service_type

9. Estimator Accuracy: 
Accuracy of estimates (initial vs final) by estimator and ranking of estimators by highest accuracy. 

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.kpi_estimator_accuracy AS
SELECT
  e.estimator_id,
  est.estimator_name,
  COUNT(e.estimate_id) as total_estimates,
  ROUND(AVG(e.accuracy_score), 2) as avg_accuracy_score,
  ROUND(AVG(e.variance_pct), 2) as avg_variance_pct,
  SUM(CASE WHEN ABS(e.variance_pct) <= 10 THEN 1 ELSE 0 END) as within_10pct_count,
  ROUND((SUM(CASE WHEN ABS(e.variance_pct) <= 10 THEN 1 ELSE 0 END) / COUNT(e.estimate_id)) * 100, 2) as within_10pct_rate,
  RANK() OVER (ORDER BY AVG(e.accuracy_score) DESC) as accuracy_rank,
  current_timestamp() as created_at
FROM automobile_repair.gold.fact_estimate e
JOIN automobile_repair.gold.dim_estimator est ON e.estimator_id = est.estimator_id
WHERE e.accuracy_score IS NOT NULL
GROUP BY e.estimator_id, est.estimator_name
ORDER BY avg_accuracy_score DESC

10. Technician Workload: 
Number of orders handled and total days in shop per technician per month and ranking of technicians by workload

In [0]:
%sql
CREATE OR REPLACE TABLE automobile_repair.gold.kpi_technician_workload AS
SELECT
  f.technician_id,
  t.technician_name,
  f.store_id,
  s.store_name,
  f.order_year,
  f.order_month,
  COUNT(f.order_id) as orders_handled,
  SUM(f.days_in_shop) as total_days_worked,
  ROUND(AVG(f.days_in_shop), 2) as avg_days_per_order,
  ROUND(AVG(f.delivery_accuracy_days), 2) as avg_delivery_accuracy,
  current_timestamp() as created_at
FROM automobile_repair.gold.fact_order_details f
JOIN automobile_repair.gold.dim_technician t ON f.technician_id = t.technician_id
JOIN automobile_repair.gold.dim_store s ON f.store_id = s.store_id
WHERE f.days_in_shop IS NOT NULL
GROUP BY f.technician_id, t.technician_name, f.store_id, s.store_name, f.order_year, f.order_month
ORDER BY f.order_year DESC, f.order_month DESC, orders_handled DESC